# CineScore Phase 3B: NLP Micro-Patch (37 Row Diagnostic)
This notebook verifies and repairs the 37 rows in `v8_FINAL_CINESCORE_NLP.csv` that fell back to failsafe values (5.0/5.5). It joins the narrative strings from `v6` to run a high-fidelity extraction using Groq's `llama3-8b` model.

In [ ]:
%pip install -q groq python-dotenv tqdm

In [ ]:
import pandas as pd
import numpy as np
import time
import json
import os
from groq import Groq
from tqdm.notebook import tqdm

# --- 1. ENVIRONMENT & PATHING ---
if os.path.exists('/content'):
    from google.colab import userdata, drive
    drive.mount('/content/drive')
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    BASE_PATH = '/content/drive/MyDrive/CineScore'
    V8_PATH = f'{BASE_PATH}/v8_FINAL_CINESCORE_NLP.csv'
    V6_PATH = f'{BASE_PATH}/v6_master_analytical_df.csv'
else:
    from dotenv import load_dotenv
    load_dotenv()
    GROQ_API_KEY = os.getenv('GROQ_API_KEY')
    V8_PATH = '../Data/Processed_Dataset/v8_FINAL_CINESCORE_NLP.csv'
    V6_PATH = '../Data/Processed_Dataset/v6_master_analytical_df.csv'

print(f'✅ System Initialized. Target: {V8_PATH}')

In [ ]:
# --- 2. DATA RECOVERY (JOINING OVERVIEWS) ---
print('📥 Loading datasets...')
df_v8 = pd.read_csv(V8_PATH)
df_v6 = pd.read_csv(V6_PATH)[['id', 'overview']]

# Merge to get overviews back for the patch session
df = df_v8.merge(df_v6, on='id', how='left')

# Isolate failsafe rows
patch_mask = df['four_quadrant_appeal'].isin([5.0, 5.5])
rows_to_patch = df[patch_mask].copy()

print(f'📈 Found {len(rows_to_patch)} rows requiring verification (5.0 or 5.5 failsafes).')

In [ ]:
# --- 3. GROQ ENGINE SETUP ---
client = Groq(api_key=GROQ_API_KEY)
SYSTEM_PROMPT = """
You are a Senior Hollywood Studio Executive calculating risk matrices for greenlighting scripts.
Evaluate the user's plot overview strictly on two commercial metrics on a 1.0 to 10.0 scale.

Metric 1 - "four_quadrant_appeal": Does this appeal uniformly to all demographics (kids, teens, adults, seniors), or is it deeply niche, complex, or R-rated? (Higher score = broader audience).
Metric 2 - "high_concept_marketability": Is the plot highly original, easy to pitch in 5 words, and extremely easy to sell on a billboard? (Higher score = instantly understandable hook).

OUTPUT RULES:
1. Return ONLY a valid JSON object.
2. Values must be floats with one decimal place.
"""

def patch_row(overview):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': f'Plot Overview: "{overview}"'}
            ],
            model='llama3-8b-8192',
            temperature=0.0,
            response_format={'type': 'json_object'}
        )
        return json.loads(chat_completion.choices[0].message.content)
    except Exception as e:
        return {'error': str(e)}

In [ ]:
# --- 4. DIAGNOSTIC EXECUTION LOOP ---
patch_count = 0
bypass_count = 0

for idx in tqdm(rows_to_patch.index, desc='Patching'):
    row = df.loc[idx]
    movie_id = row['id']
    title = row['title']
    overview = str(row['overview']) if pd.notna(row['overview']) else ''

    if len(overview) > 15:
        print(f'✅ Row [{movie_id} - {title}]: Valid Plot ({len(overview)} chars). Extracting...')
        
        result = patch_row(overview)
        
        if 'error' not in result:
            df.at[idx, 'four_quadrant_appeal'] = float(result.get('four_quadrant_appeal', 5.0))
            df.at[idx, 'high_concept_marketability'] = float(result.get('high_concept_marketability', 5.0))
            patch_count += 1
        else:
            print(f'   ⚠️ API Error: {result["error"]}')
        
        time.sleep(2.1) # Respect Rate Limits
    else:
        print(f'❌ Row [{movie_id} - {title}]: No TMDB plot available. Bypassing.')
        bypass_count += 1

print(f'\n🔥 PATCHING SESSION FINISHED')
print(f'Upgraded: {patch_count} | Verified Gaps: {bypass_count}')

In [ ]:
# --- 5. CLEAN EXPORT ---
df_final = df.drop(columns=['overview'])
df_final.to_csv(V8_PATH, index=False)
print(f'🚀 Perfectly patched dataset secured to: {V8_PATH}')